In [1]:
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# from keras.datasets import cifar10
# from keras.utils import to_categorical
# # load dataset
# (X_train, y_train), (X_test, y_test) = cifar10.load_data()
# # one hot encode target values
# y_train = to_categorical(y_train)
# y_test = to_categorical(y_test)

# X_train = X_train.astype('float32')
# X_test = X_test.astype('float32')
# # normalize to range 0-1
# X_train = X_train / 255.0
# X_test = X_test / 255.0


# np.save('train_data.npy', X_train)
# np.save('train_labels.npy', y_train)
# np.save('test_data.npy', X_test)
# np.save('test_labels.npy', y_test)

In [3]:
def get_train_data():
    return np.load('train_data.npy'), np.load('train_labels.npy')

def get_test_data():
    return np.load('test_data.npy'), np.load('test_labels.npy')

In [4]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=(3, 3), padding=1)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=(3, 3), padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2))
        self.dropout1 = nn.Dropout2d(p=0.2)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=(3, 3), padding=1)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=(3, 3), padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2))
        self.dropout2 = nn.Dropout2d(p=0.2)
        self.conv5 = nn.Conv2d(64, 128, kernel_size=(3, 3), padding=1)
        self.conv6 = nn.Conv2d(128, 128, kernel_size=(3, 3), padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=(2, 2))
        self.dropout3 = nn.Dropout2d(p=0.2)
        self.flatten1 = nn.Flatten()
        self.fc1 = nn.Linear(2048, 128)
        self.dropout4 = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool1(x)
        x = self.dropout1(x)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)
        x = self.dropout2(x)
        x = F.relu(self.conv5(x))
        x = F.relu(self.conv6(x))
        x = self.pool3(x)
        x = self.dropout3(x)
        x = self.flatten1(x)
        x = F.relu(self.fc1(x))
        x = self.dropout4(x)
        x = self.fc2(x)
        return x

In [5]:
def train_model(model, X_train, y_train, loss_fn, optimizer, epochs, batch_size, device=torch.device('cpu')):
    # Convert numpy arrays to PyTorch tensors
    X_train = torch.from_numpy(X_train).float()
    y_train = torch.from_numpy(y_train).long()

    # Create a TensorDataset
    train_dataset = TensorDataset(X_train, y_train)

    # Create a DataLoader for the training dataset with the defined batch size
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

    for epoch in range(epochs):
        total_correct = 0
        total_samples = 0
        total_loss = 0

        for i, (data, labels) in enumerate(train_loader):
            data = data.to(device)
            labels = labels.to(device)
            # Forward pass
            outputs = model(data)
            loss = loss_fn(outputs, labels)

            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Compute training accuracy
            _, predicted = torch.max(outputs.data, 1)
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)
            total_loss += loss.item()

        # Print loss and accuracy at the end of the epoch
        epoch_loss = total_loss / (i + 1)
        epoch_accuracy = total_correct / total_samples
        print('Epoch [{}/{}], Loss: {:.4f}, Accuracy: {:.4f}'
            .format(epoch+1, epochs, epoch_loss, epoch_accuracy))

    return model

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Availabe device for training is: {device}')

Availabe device for training is: cuda


In [7]:
X_train, y_train = get_train_data()
X_train = np.transpose(X_train, (0, 3, 1, 2))
y_train = np.argmax(y_train, axis=1)
print(X_train.shape)

(50000, 3, 32, 32)


In [8]:
model = Model().to(device)
config = {
    "mode": {
      "type": "local",
      "params": {}
      
    },
  "temp_data_path": "../../../../",
    "lib": "pytorch",
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 70,
    "batch_size": 64,
    "loss": nn.CrossEntropyLoss(),
    "optimizer": optim.RMSprop(model.parameters(), lr=0.001)
}
model = train_model(model, X_train, y_train, config["loss"], config["optimizer"], config["epochs"], config["batch_size"], device)

Epoch [1/70], Loss: 1.8715, Accuracy: 0.3170
Epoch [2/70], Loss: 1.4398, Accuracy: 0.4763
Epoch [3/70], Loss: 1.2282, Accuracy: 0.5612
Epoch [4/70], Loss: 1.0813, Accuracy: 0.6171
Epoch [5/70], Loss: 0.9764, Accuracy: 0.6547
Epoch [6/70], Loss: 0.9001, Accuracy: 0.6850
Epoch [7/70], Loss: 0.8336, Accuracy: 0.7082
Epoch [8/70], Loss: 0.7866, Accuracy: 0.7226
Epoch [9/70], Loss: 0.7434, Accuracy: 0.7380
Epoch [10/70], Loss: 0.7070, Accuracy: 0.7520
Epoch [11/70], Loss: 0.6764, Accuracy: 0.7623
Epoch [12/70], Loss: 0.6526, Accuracy: 0.7734
Epoch [13/70], Loss: 0.6258, Accuracy: 0.7798
Epoch [14/70], Loss: 0.6024, Accuracy: 0.7902
Epoch [15/70], Loss: 0.5853, Accuracy: 0.7939
Epoch [16/70], Loss: 0.5673, Accuracy: 0.8016
Epoch [17/70], Loss: 0.5540, Accuracy: 0.8031
Epoch [18/70], Loss: 0.5398, Accuracy: 0.8104
Epoch [19/70], Loss: 0.5278, Accuracy: 0.8152
Epoch [20/70], Loss: 0.5192, Accuracy: 0.8170
Epoch [21/70], Loss: 0.5035, Accuracy: 0.8228
Epoch [22/70], Loss: 0.4955, Accuracy: 0.82

In [9]:
def evaluate_model(model, X_test, y_test, batch_size, device=torch.device('cpu')):
    # Convert numpy arrays to PyTorch tensors
    X_test = torch.from_numpy(X_test).float()
    y_test = torch.from_numpy(y_test).long()

    # Create a TensorDataset
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    total_correct = 0
    total_samples = 0

    for i, (data, labels) in enumerate(test_loader):
        data = data.to(device)
        labels = labels.to(device)
        # Forward pass
        outputs = model(data)

        # Compute training accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)

    return total_correct / total_samples

In [10]:
X_test, y_test = get_test_data()
X_test = np.transpose(X_test, (0, 3, 1, 2))
y_test = np.argmax(y_test, axis=1)

In [11]:
acc = evaluate_model(model, X_test, y_test, config["batch_size"], device)
print("\nTest accuracy: %.1f%%" % (100.0 * acc))


Test accuracy: 73.5%
